In [2]:
from typing import TypedDict, Literal 

from openai import OpenAI

from langgraph.graph import StateGraph, START, END 

In [3]:
from groq import Groq

client = Groq()

In [4]:
class AgentState(TypedDict):
    query: str 
    route: str 
    confidence: float 
    response: str 
    probabilities: dict[str, float]

In [5]:
def billing_node(state: AgentState):
    return {
        "response": "Billing agent is handling the query",
    }

In [6]:
def technical_node(state: AgentState):
    return {
        "response": "Technical agent is handling the query",
    }

In [7]:
def shipping_node(state: AgentState):
    return {
        "response": "Shipping agent is handling the query",
    }

In [8]:
def human_node(state: AgentState):
    return {
        "response": "Human agent is handling the query",
    }

In [ ]:
## OpenAI Way
def router_node(state: AgentState):
    decision = client.decisions.create(
        model="gpt-6-luna",
        input=state["query"], # query is the user's query
        questions=[
            {
                "type": "choice",
                "name": "support_route",
                "instructions": (
                    "Which support team should handle this customer request?"
                    "Choose the most appropriate option from the following choices:"
                ),
                "choices": [
                    {
                        "value": "billing",
                        "description": "For payments, refunds, invoices, billing and account issues"
                    },
                    {
                        "value": "technical",
                        "description": "For bugs, product issues, feature requests, auhentication and login issues, technical support or platform issues"
                    },
                    {
                        "value": "shipping",
                        "description": "For shipping, tracking, delivery, returns, logistics or order issues"
                    },
                    {
                        "value": "other",
                        "description": "For complex issues that require human intervention or escalation or if anything is unclear"
                    }
                ]
            }
        ]
    )

    answer = decision.answers[0]
    if answer.type != "choice":
        return {
            "route": "human",
            "confidence": 0.0,
            "probabilities": {}
        }
    
    probabilities = {
        item.value : item.probability 
        for item in answer.probabilities
    }

    route = answer.choice 
    if answer.confidence < 0.75 or route == "other":
        route = "human"
    
    return {
        "route": route,
        "confidence": answer.confidence,
        "probabilities": probabilities
    }

In [9]:
from typing import Dict, Literal
from pydantic import BaseModel, Field
from pydantic import BaseModel


from langchain_groq import ChatGroq

SupportChoice = Literal["billing", "technical", "shipping", "other"]


class RouteEvaluation(BaseModel):
    choice: SupportChoice = Field(
        description=(
            "Most appropriate support team: 'billing' (payments, refunds, invoices, billing, account issues), "
            "'technical' (bugs, product issues, feature requests, authentication/login, technical/platform issues), "
            "'shipping' (shipping, tracking, delivery, returns, logistics, order issues), or "
            "'other' (complex issues needing human escalation or unclear requests)."
        )
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score between 0.0 and 1.0 for the selected choice.",
    )
    probabilities: Dict[SupportChoice, float] = Field(
        description="Estimated probability distribution across billing, technical, shipping, and other summing to ~1.0."
    )


router_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
).with_structured_output(RouteEvaluation)


def router_node(state: AgentState):
    try:
        result: RouteEvaluation = router_llm.invoke(
            [
                (
                    "system",
                    "Which support team should handle this customer request? "
                    "Choose the most appropriate option from: billing, technical, shipping, other. "
                    "Also provide a calibrated confidence score (0.0 to 1.0) and probabilities for each choice.",
                ),
                ("human", state["query"]),
            ]
        )
    except Exception:
        return {
            "route": "human",
            "confidence": 0.0,
            "probabilities": {},
        }

    route: str = result.choice
    if result.confidence < 0.75 or route == "other":
        route = "human"

    return {
        "route": route,
        "confidence": result.confidence,
        "probabilities": dict(result.probabilities),
    }

In [ ]:
# builder = StateGraph(AgentState)

# builder.add_node("router", router_node)
# builder.add_node("billing", billing_node)
# builder.add_node("technical", technical_node)
# builder.add_node("shipping", shipping_node)
# builder.add_node("human", human_node)


# builder.add_edge(START, "router") 
# builder.add_conditional_edges(
#     "router", # starting node
#     choose_next_node, # function that will decide the next node based on the decisions api
#     {
#         "billing": "billing",
#         "technical": "technical",
#         "shipping": "shipping",
#         "human": "human",
#     }

# )

# for node in ["billing", "technical", "shipping", "human"]:
#     builder.add_edge(node, END)

# graph = builder.compile()

NameError: name 'choose_next_node' is not defined

In [11]:
ALLOWED_ROUTES = {"billing", "technical", "shipping", "human"}


def choose_next_node(state: AgentState) -> str:
    route = state.get("route", "human")
    return route if route in ALLOWED_ROUTES else "human"

In [ ]:
initial_state = {
    "query": "I need help with my billing"
}


In [ ]:
graph.invoke(initial_state)

NameError: name 'graph' is not defined

In [14]:
final_state = graph.invoke({"query": "I need help with my billing"})
print(final_state["route"], final_state["confidence"])

NameError: name 'graph' is not defined

In [ ]:
graph.invoke({
    "query": "I am not able to login to my account"
})

In [ ]:
graph.invoke({
    "query": "I am not able to connect to the delivery agent"
})

In [15]:
from typing import Dict, Literal, TypedDict

from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

SupportChoice = Literal["billing", "technical", "shipping", "other"]


class RouteEvaluation(BaseModel):
    choice: SupportChoice = Field(
        description=(
            "Most appropriate support team: 'billing' (payments, refunds, invoices, billing, account issues), "
            "'technical' (bugs, product issues, feature requests, authentication/login, technical/platform issues), "
            "'shipping' (shipping, tracking, delivery, returns, logistics, order issues), or "
            "'other' (complex issues needing human escalation or unclear requests)."
        )
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score between 0.0 and 1.0 for the selected choice.",
    )
    probabilities: Dict[SupportChoice, float] = Field(
        description="Estimated probability distribution across billing, technical, shipping, and other."
    )


class AgentState(TypedDict):
    query: str
    route: str
    confidence: float
    response: str
    probabilities: dict[str, float]


router_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
).with_structured_output(RouteEvaluation)


def billing_node(state: AgentState):
    return {
        "response": "Billing agent is handling the query",
    }


def technical_node(state: AgentState):
    return {
        "response": "Technical agent is handling the query",
    }


def shipping_node(state: AgentState):
    return {
        "response": "Shipping agent is handling the query",
    }


def human_node(state: AgentState):
    return {
        "response": "Human agent is handling the query",
    }


def router_node(state: AgentState):
    try:
        result: RouteEvaluation = router_llm.invoke(
            [
                (
                    "system",
                    "Which support team should handle this customer request? "
                    "Choose the most appropriate option from: billing, technical, shipping, other. "
                    "Also provide a confidence score (0.0 to 1.0) and probabilities for each choice.",
                ),
                ("human", state["query"]),
            ]
        )
    except Exception:
        return {
            "route": "human",
            "confidence": 0.0,
            "probabilities": {},
        }

    route: str = result.choice
    if result.confidence < 0.75 or route == "other":
        route = "human"

    return {
        "route": route,
        "confidence": result.confidence,
        "probabilities": dict(result.probabilities),
    }


def choose_next_node(state: AgentState):
    return state["route"]


builder = StateGraph(AgentState)

builder.add_node("router", router_node)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_node("shipping", shipping_node)
builder.add_node("human", human_node)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    choose_next_node,
    {
        "billing": "billing",
        "technical": "technical",
        "shipping": "shipping",
        "human": "human",
    },
)

for node in ["billing", "technical", "shipping", "human"]:
    builder.add_edge(node, END)

graph = builder.compile()

queries = [
    {"query": "I need help with my billing"},
    {"query": "I am not able to login to my account"},
    {"query": "I am not able to connect to the delivery agent"},
]

for item in queries:
    result = graph.invoke(item)
    print(item["query"], "->", result["route"], f"({result['confidence']:.2f})", "|", result["response"])

I need help with my billing -> billing (0.95) | Billing agent is handling the query
I am not able to login to my account -> technical (0.95) | Technical agent is handling the query
I am not able to connect to the delivery agent -> technical (0.85) | Technical agent is handling the query
